# Refinement Model — Learned Post-Processing

Train a lightweight 3D U-Net that takes raw probability maps from the main SegResNet
and outputs clean segmentations. Replaces hysteresis thresholding, morphological closing,
and dust removal with a single learned forward pass.

**Input:** 1ch probability map (320^3 float16) from v9 Gaussian SWI inference

**Output:** 1ch refined logits (320^3) → sigmoid → threshold 0.5 → binary mask

**Training data:** Pre-computed probmaps at `/data/refinement_data/probmaps/` + GT labels

**Phases:**
- Phase 1: 0.5*BCE + 0.5*Dice (learn basic refinement)
- Phase 2: 0.2*BCE + 0.2*Dice + 0.3*clDice + 0.3*Boundary (topology-aware refinement)

**Acceptance criterion:** Beats hand-tuned post-processing (hysteresis+closing+dust) on val set

## 1. Imports & Config

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
import tifffile
from pathlib import Path
import matplotlib.pyplot as plt
from fastai.learner import Learner
from fastai.callback.schedule import fit_one_cycle, fit_flat_cos
from fastai.callback.tracker import SaveModelCallback
from fastai.callback.fp16 import MixedPrecision
from fastai.data.core import DataLoaders
from fastai.metrics import Metric
from scipy.ndimage import distance_transform_edt, label as scipy_label
from scipy.ndimage import binary_closing, generate_binary_structure, binary_propagation
from scipy.ndimage import zoom as scipy_zoom
from topometrics.leaderboard import compute_leaderboard_score
import random
import time

# Config
ROOT = Path("/home/mongomatt/Projects/vesuvius")
TRAIN_LBL = ROOT / "train_labels"
PROBMAP_DIR = Path("/data/refinement_data/probmaps")
CKPT_DIR = ROOT / "checkpoints"
CKPT_DIR.mkdir(exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
VOL_SIZE = 320         # Full volume size (used at inference/metric eval only)
PATCH_SIZE = 160       # Train on 160^3 random crops (~2.5 GB VRAM; 320^3 needs ~20 GB)
BATCH_SIZE = 2         # 160^3 patches fit bs=2 easily
NUM_WORKERS = 4
EPOCHS = 50
LR = 1e-3              # From scratch, higher LR appropriate
SEED = 42
N_VAL_VOLUMES = 5      # For competition metric eval
METRIC_DOWNSAMPLE = 4  # 320 -> 80 for fast Betti matching

# Hand-tuned baseline thresholds (from eval_inference.py sweep)
BASELINE_T_LOW = 0.35
BASELINE_T_HIGH = 0.80
CLOSING_Z_RADIUS = 2
CLOSING_XY_RADIUS = 1
DUST_MIN_SIZE = 100

torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

print(f"Device: {DEVICE}")
print(f"CUDA: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'}")
print(f"Training: {PATCH_SIZE}^3 patches, bs={BATCH_SIZE} (~2.5 GB VRAM)")
print(f"Inference: full {VOL_SIZE}^3 volumes (forward only, ~1 GB)")
print(f"Probmap dir: {PROBMAP_DIR}")
print(f"Epochs: {EPOCHS}, LR: {LR}")

## 2. Data Exploration

In [ ]:
# Check available probmaps
probmap_files = sorted(PROBMAP_DIR.glob("*.npy"))
probmap_ids = set(int(p.stem) for p in probmap_files)
print(f"Available probmaps: {len(probmap_ids)}")

# Check available labels
label_files = sorted(TRAIN_LBL.glob("*.tif"))
label_ids = set(int(p.stem) for p in label_files)
print(f"Available labels: {len(label_ids)}")

# Intersection: volumes with both probmap and label
available_ids = probmap_ids & label_ids
print(f"Volumes with both probmap + label: {len(available_ids)}")

# Load and inspect a sample probmap
sample_id = sorted(available_ids)[0]
sample_prob = np.load(PROBMAP_DIR / f"{sample_id}.npy")
sample_lbl = tifffile.imread(TRAIN_LBL / f"{sample_id}.tif")

print(f"\nSample {sample_id}:")
print(f"  Probmap: shape={sample_prob.shape}, dtype={sample_prob.dtype}, "
      f"range=[{sample_prob.min():.4f}, {sample_prob.max():.4f}]")
print(f"  Label:   shape={sample_lbl.shape}, dtype={sample_lbl.dtype}, "
      f"unique={np.unique(sample_lbl)}")

In [ ]:
# Visualize probmap vs label
mid = VOL_SIZE // 2
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

# Row 1: probmap slices
axes[0, 0].imshow(sample_prob[mid], cmap="hot", vmin=0, vmax=1)
axes[0, 0].set_title(f"Probmap axial z={mid}")
axes[0, 1].imshow(sample_prob[:, mid], cmap="hot", vmin=0, vmax=1)
axes[0, 1].set_title(f"Probmap coronal y={mid}")
axes[0, 2].imshow(sample_prob[:, :, mid], cmap="hot", vmin=0, vmax=1)
axes[0, 2].set_title(f"Probmap sagittal x={mid}")

# Row 2: label slices (0=bg, 1=fg, 2=unlabeled)
for i, (slc_prob, slc_lbl, title) in enumerate([
    (sample_prob[mid], sample_lbl[mid], f"Overlay z={mid}"),
    (sample_prob[:, mid], sample_lbl[:, mid], f"Overlay y={mid}"),
    (sample_prob[:, :, mid], sample_lbl[:, :, mid], f"Overlay x={mid}"),
]):
    rgb = np.stack([slc_prob.astype(np.float32)]*3, axis=-1)
    rgb[slc_lbl == 1] = [0, 1, 0]  # GT foreground = green
    rgb[slc_lbl == 2] = [0, 0, 0.3]  # unlabeled = dark blue
    axes[1, i].imshow(rgb)
    axes[1, i].set_title(title)

for ax in axes.flat:
    ax.axis("off")
plt.tight_layout()
plt.show()

## 3. RefinementUNet3D Model

In [ ]:
class ConvBlock3D(nn.Module):
    """Two 3x3x3 convolutions with GroupNorm and ReLU."""
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv3d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.GroupNorm(min(8, out_ch), out_ch),
            nn.ReLU(inplace=True),
            nn.Conv3d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.GroupNorm(min(8, out_ch), out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.conv(x)


class RefinementUNet3D(nn.Module):
    """
    Shallow 3D U-Net for refining probability maps into clean segmentations.
    
    Channels: [8, 16, 32, 64] — 4 encoder levels.
    Concat skip connections, trilinear upsampling.
    Dropout at bottleneck.
    
    Training: 160^3 patches (~2.5 GB VRAM at bs=2)
    Inference: full 320^3 volumes (forward only, ~1 GB)
    """
    def __init__(self, in_ch=1, out_ch=1, channels=(8, 16, 32, 64), dropout=0.2):
        super().__init__()
        
        # Encoder
        self.enc1 = ConvBlock3D(in_ch, channels[0])
        self.enc2 = ConvBlock3D(channels[0], channels[1])
        self.enc3 = ConvBlock3D(channels[1], channels[2])
        
        # Bottleneck
        self.bottleneck = ConvBlock3D(channels[2], channels[3])
        self.dropout = nn.Dropout3d(p=dropout)
        
        # Decoder (input channels = upsampled + skip)
        self.dec3 = ConvBlock3D(channels[3] + channels[2], channels[2])
        self.dec2 = ConvBlock3D(channels[2] + channels[1], channels[1])
        self.dec1 = ConvBlock3D(channels[1] + channels[0], channels[0])
        
        # Final 1x1 conv
        self.conv_final = nn.Conv3d(channels[0], out_ch, 1)
        
        # Pooling
        self.pool = nn.MaxPool3d(2)

    def forward(self, x):
        # Encoder
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        
        # Bottleneck
        b = self.bottleneck(self.pool(e3))
        b = self.dropout(b)
        
        # Decoder with skip connections
        d3 = F.interpolate(b, size=e3.shape[2:], mode='trilinear', align_corners=False)
        d3 = self.dec3(torch.cat([d3, e3], dim=1))
        
        d2 = F.interpolate(d3, size=e2.shape[2:], mode='trilinear', align_corners=False)
        d2 = self.dec2(torch.cat([d2, e2], dim=1))
        
        d1 = F.interpolate(d2, size=e1.shape[2:], mode='trilinear', align_corners=False)
        d1 = self.dec1(torch.cat([d1, e1], dim=1))
        
        return self.conv_final(d1)


# Test forward pass
model = RefinementUNet3D()
n_params = sum(p.numel() for p in model.parameters())
print(f"RefinementUNet3D: {n_params:,} parameters ({n_params/1e6:.2f}M)")

# Test at training size (160^3)
with torch.no_grad():
    dummy = torch.randn(1, 1, PATCH_SIZE, PATCH_SIZE, PATCH_SIZE)
    out = model(dummy)
    print(f"Training:  {dummy.shape} -> {out.shape}")

# Test at inference size (320^3)
with torch.no_grad():
    dummy = torch.randn(1, 1, VOL_SIZE, VOL_SIZE, VOL_SIZE)
    out = model(dummy)
    print(f"Inference: {dummy.shape} -> {out.shape}")

del model, dummy, out
torch.cuda.empty_cache()

## 4. Dataset

In [ ]:
def compute_signed_distance_map(label, mask):
    """
    Compute signed distance transform from ground truth boundary.
    Negative inside foreground, positive outside, zero at boundary.
    Normalized by max distance so values are roughly in [-1, 1].
    """
    fg = label.astype(bool)
    bg = ~fg
    dist_inside = distance_transform_edt(fg)
    dist_outside = distance_transform_edt(bg)
    signed_dist = dist_outside - dist_inside
    max_dist = max(np.abs(signed_dist).max(), 1.0)
    signed_dist = signed_dist / max_dist
    signed_dist = signed_dist * mask
    return signed_dist.astype(np.float32)


class RefinementDataset(Dataset):
    """
    Dataset for refinement model training.
    
    Loads pre-computed probability maps (.npy float16) and GT labels (.tif uint8).
    Training: random 160^3 crops from 320^3 volumes (with augmentations).
    Validation: random 160^3 crops (no augmentation) — full-volume eval done in metric.
    
    Returns:
        probmap: (1, PS, PS, PS) float32 — model input
        (label, mask, dist_map): tuple of (1, PS, PS, PS) tensors — targets
    """
    def __init__(self, ids, probmap_dir, lbl_dir, patch_size=160,
                 augment=False, compute_dist=True):
        self.ids = ids
        self.probmap_dir = Path(probmap_dir)
        self.lbl_dir = Path(lbl_dir)
        self.patch_size = patch_size
        self.augment = augment
        self.compute_dist = compute_dist

    def __len__(self):
        return len(self.ids)

    def _random_crop(self, *arrays):
        """Extract random patch_size^3 crop from all arrays at the same position."""
        ps = self.patch_size
        d, h, w = arrays[0].shape
        z = random.randint(0, d - ps)
        y = random.randint(0, h - ps)
        x = random.randint(0, w - ps)
        return [a[z:z+ps, y:y+ps, x:x+ps] for a in arrays]

    def __getitem__(self, idx):
        vol_id = self.ids[idx]

        # Load probmap (float16 -> float32)
        prob = np.load(self.probmap_dir / f"{vol_id}.npy").astype(np.float32)

        # Load GT label
        lbl = tifffile.imread(self.lbl_dir / f"{vol_id}.tif")

        # Mask: valid where label is 0 or 1
        mask = (lbl != 2).astype(np.float32)
        label = (lbl == 1).astype(np.float32)

        # Signed distance map for boundary loss (compute on full volume, then crop)
        if self.compute_dist:
            dist_map = compute_signed_distance_map(label, mask)
        else:
            dist_map = np.zeros_like(label)

        # Random crop to patch_size^3
        prob, label, mask, dist_map = self._random_crop(prob, label, mask, dist_map)

        # Augmentations: random flips + 90-degree rotations
        if self.augment:
            for axis in range(3):
                if random.random() > 0.5:
                    prob = np.flip(prob, axis=axis).copy()
                    label = np.flip(label, axis=axis).copy()
                    mask = np.flip(mask, axis=axis).copy()
                    dist_map = np.flip(dist_map, axis=axis).copy()

            k = random.randint(0, 3)
            plane_axes = random.choice([(0, 1), (0, 2), (1, 2)])
            if k > 0:
                prob = np.rot90(prob, k=k, axes=plane_axes).copy()
                label = np.rot90(label, k=k, axes=plane_axes).copy()
                mask = np.rot90(mask, k=k, axes=plane_axes).copy()
                dist_map = np.rot90(dist_map, k=k, axes=plane_axes).copy()

        # Add channel dim: (1, D, H, W)
        prob = torch.from_numpy(prob).unsqueeze(0)
        label = torch.from_numpy(label).unsqueeze(0)
        mask = torch.from_numpy(mask).unsqueeze(0)
        dist_map = torch.from_numpy(dist_map).unsqueeze(0)

        return prob, (label, mask, dist_map)


# Quick test
test_ds = RefinementDataset([sample_id], PROBMAP_DIR, TRAIN_LBL,
                             patch_size=PATCH_SIZE, augment=True, compute_dist=True)
x, (y, m, d) = test_ds[0]
print(f"Probmap: {x.shape}, range=[{x.min():.3f}, {x.max():.3f}]")
print(f"Label:   {y.shape}, sum={y.sum().item():.0f}")
print(f"Mask:    {m.shape}, coverage={m.mean().item()*100:.1f}%")
print(f"DistMap: {d.shape}, range=[{d.min():.3f}, {d.max():.3f}]")

## 5. DataLoaders

In [ ]:
# Split by scroll_id: hold out scroll 26002 for validation (same as main model)
train_df = pd.read_csv(ROOT / "train.csv")

# Only keep IDs that have both probmap and label
available_ids = set(int(p.stem) for p in PROBMAP_DIR.glob("*.npy")) & \
                set(int(p.stem) for p in TRAIN_LBL.glob("*.tif"))
train_df = train_df[train_df.id.isin(available_ids)].reset_index(drop=True)

VAL_SCROLL = 26002
val_ids = train_df[train_df.scroll_id == VAL_SCROLL].id.tolist()
trn_ids = train_df[train_df.scroll_id != VAL_SCROLL].id.tolist()

print(f"Total available: {len(train_df)} volumes")
print(f"Train: {len(trn_ids)}, Val: {len(val_ids)}")
print(f"Val scroll: {VAL_SCROLL}")

# Phase 1: no distance map needed (BCE + Dice only)
trn_ds = RefinementDataset(trn_ids, PROBMAP_DIR, TRAIN_LBL,
                            patch_size=PATCH_SIZE, augment=True, compute_dist=False)
val_ds = RefinementDataset(val_ids, PROBMAP_DIR, TRAIN_LBL,
                            patch_size=PATCH_SIZE, augment=False, compute_dist=False)

trn_dl = DataLoader(trn_ds, batch_size=BATCH_SIZE, shuffle=True,
                    num_workers=NUM_WORKERS, pin_memory=True)
val_dl = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                    num_workers=NUM_WORKERS, pin_memory=True)

dls = DataLoaders(trn_dl, val_dl, device=DEVICE)
print(f"DataLoaders ready (Phase 1: {PATCH_SIZE}^3 patches, compute_dist=False).")

## 6. Loss Functions & Metrics

In [ ]:
# --- clDice: topology-preserving loss via soft skeletons ---

def soft_erode_3d(img):
    return -F.max_pool3d(-img, kernel_size=3, stride=1, padding=1)

def soft_dilate_3d(img):
    return F.max_pool3d(img, kernel_size=3, stride=1, padding=1)

def soft_open_3d(img):
    return soft_dilate_3d(soft_erode_3d(img))

def soft_skeleton_3d(img, iters=5):
    skel = F.relu(img - soft_open_3d(img))
    for _ in range(iters):
        img = soft_erode_3d(img)
        delta = F.relu(img - soft_open_3d(img))
        skel = skel + delta
    return skel

def soft_cldice(pred_sig, target, iters=5, smooth=1.0):
    """clDice on 2x-downsampled volumes for memory efficiency."""
    pred_ds = F.avg_pool3d(pred_sig, kernel_size=2)
    with torch.no_grad():
        target_ds = F.avg_pool3d(target, kernel_size=2)
    skel_pred = soft_skeleton_3d(pred_ds, iters=iters)
    with torch.no_grad():
        skel_target = soft_skeleton_3d(target_ds, iters=iters)
    tprec = ((skel_pred * target_ds).sum() + smooth) / (skel_pred.sum() + smooth)
    tsens = ((skel_target * pred_ds).sum() + smooth) / (skel_target.sum() + smooth)
    return 2.0 * tprec * tsens / (tprec + tsens + 1e-8)


def boundary_loss(pred_sig, dist_map, mask):
    """Boundary loss: push predicted surface toward GT surface."""
    return (pred_sig * dist_map * mask).sum() / (mask.sum() + 1e-8)


class Phase1Loss(nn.Module):
    """0.5*BCE + 0.5*Dice (masked). Simple baseline for initial training."""
    def __init__(self, smooth=1.0):
        super().__init__()
        self.smooth = smooth

    def forward(self, pred, target_tuple):
        target, mask, _dist_map = target_tuple
        pred_sig = torch.sigmoid(pred)
        pred_masked = pred_sig * mask
        target_masked = target * mask

        bce = F.binary_cross_entropy_with_logits(pred, target, reduction='none')
        bce = (bce * mask).sum() / (mask.sum() + 1e-8)

        intersection = (pred_masked * target_masked).sum()
        dice = (2.0 * intersection + self.smooth) / (
            pred_masked.sum() + target_masked.sum() + self.smooth
        )
        dice_loss = 1.0 - dice

        return 0.5 * bce + 0.5 * dice_loss


class Phase2Loss(nn.Module):
    """0.2*BCE + 0.2*Dice + 0.3*clDice + 0.3*Boundary (masked)."""
    def __init__(self, smooth=1.0, skel_iters=5):
        super().__init__()
        self.smooth = smooth
        self.skel_iters = skel_iters

    def forward(self, pred, target_tuple):
        target, mask, dist_map = target_tuple
        pred_sig = torch.sigmoid(pred)
        pred_masked = pred_sig * mask
        target_masked = target * mask

        bce = F.binary_cross_entropy_with_logits(pred, target, reduction='none')
        bce = (bce * mask).sum() / (mask.sum() + 1e-8)

        intersection = (pred_masked * target_masked).sum()
        dice = (2.0 * intersection + self.smooth) / (
            pred_masked.sum() + target_masked.sum() + self.smooth
        )
        dice_loss = 1.0 - dice

        cldice_loss = 1.0 - soft_cldice(pred_masked, target_masked,
                                         iters=self.skel_iters, smooth=self.smooth)
        bd_loss = boundary_loss(pred_sig, dist_map, mask)

        return 0.2 * bce + 0.2 * dice_loss + 0.3 * cldice_loss + 0.3 * bd_loss


class MaskedDiceMetric(Metric):
    """Dice coefficient metric (masked) for fastai."""
    def __init__(self):
        self.reset()

    def reset(self):
        self.intersection = 0.0
        self.union = 0.0

    def accumulate(self, learn):
        pred = torch.sigmoid(learn.pred) > 0.5
        target, mask, _dist = learn.y
        pred_m = pred * mask
        tgt_m = target * mask
        self.intersection += (pred_m * tgt_m).sum().item()
        self.union += pred_m.sum().item() + tgt_m.sum().item()

    @property
    def value(self):
        return (2.0 * self.intersection + 1.0) / (self.union + 1.0)

    @property
    def name(self):
        return "dice"


print("Phase 1 loss: 0.5*BCE + 0.5*Dice")
print("Phase 2 loss: 0.2*BCE + 0.2*Dice + 0.3*clDice + 0.3*Boundary")

In [ ]:
# --- Hand-tuned post-processing baseline (for comparison) ---

def hysteresis_threshold(prob, t_low=0.45, t_high=0.85):
    strong = prob >= t_high
    weak = prob >= t_low
    struct = generate_binary_structure(3, 3)
    return binary_propagation(strong, structure=struct, mask=weak).astype(np.uint8)

def build_anisotropic_struct(z_radius=2, xy_radius=1):
    sz = 2 * z_radius + 1
    sxy = 2 * xy_radius + 1
    struct = np.zeros((sz, sxy, sxy), dtype=bool)
    cy, cx = xy_radius, xy_radius
    for y in range(sxy):
        for x in range(sxy):
            if (y - cy)**2 + (x - cx)**2 <= xy_radius**2:
                struct[:, y, x] = True
    return struct

def postprocess(probs, t_low=0.45, t_high=0.85, z_radius=2, xy_radius=1, min_size=100):
    """Hand-tuned post-processing: hysteresis + anisotropic closing + dust removal."""
    binary = hysteresis_threshold(probs, t_low, t_high)
    struct = build_anisotropic_struct(z_radius, xy_radius)
    closed = binary_closing(binary, structure=struct)
    labeled, n_components = scipy_label(closed)
    if n_components > 0:
        component_sizes = np.bincount(labeled.ravel())
        small_mask = component_sizes < min_size
        small_mask[0] = False
        closed[small_mask[labeled]] = 0
    return closed.astype(np.uint8)


# --- Competition metric for refinement model ---

class RefinementCompetitionMetric(Metric):
    """
    Competition metric for the refinement model.
    
    Loads probmap -> refinement model forward pass -> sigmoid -> threshold 0.5 -> comp_score.
    No sliding window needed (model takes full 320^3 volume).
    """
    def __init__(self, val_ids, probmap_dir, lbl_dir, n_volumes=5, downsample=4):
        self.val_ids = val_ids[:n_volumes]
        self.probmap_dir = Path(probmap_dir)
        self.lbl_dir = Path(lbl_dir)
        self.downsample = downsample
        self._model = None
        self.reset()

    def reset(self):
        self._model = None

    def accumulate(self, learn):
        self._model = learn.model

    @property
    def value(self):
        if self._model is None:
            return 0.0

        self._model.eval()
        scores = []
        for vid in self.val_ids:
            prob = np.load(self.probmap_dir / f"{vid}.npy").astype(np.float32)
            lbl = tifffile.imread(self.lbl_dir / f"{vid}.tif")

            # Refinement model forward pass
            with torch.no_grad(), torch.amp.autocast("cuda"):
                inp = torch.from_numpy(prob).unsqueeze(0).unsqueeze(0).to(
                    next(self._model.parameters()).device
                )
                logits = self._model(inp)
                pred = (torch.sigmoid(logits).squeeze().cpu().numpy() > 0.5).astype(np.uint8)

            # Downsample for fast Betti matching
            ds = self.downsample
            if ds > 1:
                scale = 1.0 / ds
                pred = scipy_zoom(pred, scale, order=0).astype(np.uint8)
                lbl = scipy_zoom(lbl, scale, order=0).astype(np.uint8)

            report = compute_leaderboard_score(
                pred, lbl, ignore_label=2, spacing=(1, 1, 1),
                surface_tolerance=2.0, voi_alpha=0.3,
                combine_weights=(0.3, 0.35, 0.35),
            )
            scores.append(report.score)

        self._model.train()
        return float(np.mean(scores))

    @property
    def name(self):
        return "comp_score"


print("Competition metric and post-processing baseline defined.")

## 7. Phase 1 Training (BCE + Dice)

In [ ]:
model = RefinementUNet3D().to(DEVICE)
loss_fn = Phase1Loss()

comp_metric = RefinementCompetitionMetric(
    val_ids, PROBMAP_DIR, TRAIN_LBL,
    n_volumes=N_VAL_VOLUMES, downsample=METRIC_DOWNSAMPLE,
)

learn = Learner(
    dls,
    model,
    loss_func=loss_fn,
    metrics=[MaskedDiceMetric(), comp_metric],
    cbs=[
        MixedPrecision(),
        SaveModelCallback(monitor="comp_score", comp=np.greater,
                          fname="best_refinement_phase1"),
    ],
    path=CKPT_DIR,
)

print(f"Phase 1 Learner ready. Loss: 0.5*BCE + 0.5*Dice")
print(f"Metrics: dice, comp_score ({N_VAL_VOLUMES} val volumes, {METRIC_DOWNSAMPLE}x downsample)")
print(f"Schedule: fit_one_cycle (from scratch — warmup helps find good landscape)")

In [ ]:
# lr_find to validate our LR choice
from fastai.callback.schedule import valley, steep
lr_result = learn.lr_find(suggest_funcs=(valley, steep))
print(f"lr_find valley: {lr_result.valley:.2e}")
print(f"Using LR={LR} (from scratch, no pretrained weights to protect)")

In [ ]:
# Phase 1 training: fit_one_cycle (from scratch — warmup helps explore loss landscape)
print(f"Training Phase 1: {EPOCHS} epochs, LR={LR}, fit_one_cycle")
learn.fit_one_cycle(EPOCHS, lr_max=LR)

In [ ]:
# Plot Phase 1 training curves
learn.recorder.plot_loss()
plt.title("Phase 1: BCE + Dice")
plt.show()

## 8. Phase 2 Training (+ clDice + Boundary)

Continue from Phase 1 best checkpoint with topology-aware losses.
Requires distance maps, so we rebuild DataLoaders with `compute_dist=True`.

In [ ]:
# Rebuild DataLoaders with distance map computation
trn_ds2 = RefinementDataset(trn_ids, PROBMAP_DIR, TRAIN_LBL,
                             patch_size=PATCH_SIZE, augment=True, compute_dist=True)
val_ds2 = RefinementDataset(val_ids, PROBMAP_DIR, TRAIN_LBL,
                             patch_size=PATCH_SIZE, augment=False, compute_dist=True)

trn_dl2 = DataLoader(trn_ds2, batch_size=BATCH_SIZE, shuffle=True,
                     num_workers=NUM_WORKERS, pin_memory=True)
val_dl2 = DataLoader(val_ds2, batch_size=BATCH_SIZE, shuffle=False,
                     num_workers=NUM_WORKERS, pin_memory=True)

dls2 = DataLoaders(trn_dl2, val_dl2, device=DEVICE)

# Load Phase 1 best model
learn.load("best_refinement_phase1")
model = learn.model

# Create new learner with Phase 2 loss
loss_fn2 = Phase2Loss()

comp_metric2 = RefinementCompetitionMetric(
    val_ids, PROBMAP_DIR, TRAIN_LBL,
    n_volumes=N_VAL_VOLUMES, downsample=METRIC_DOWNSAMPLE,
)

learn2 = Learner(
    dls2,
    model,
    loss_func=loss_fn2,
    metrics=[MaskedDiceMetric(), comp_metric2],
    cbs=[
        MixedPrecision(),
        SaveModelCallback(monitor="comp_score", comp=np.greater,
                          fname="best_refinement_phase2"),
    ],
    path=CKPT_DIR,
)

print(f"Phase 2 Learner ready. Loss: 0.2*BCE + 0.2*Dice + 0.3*clDice + 0.3*Boundary")
print(f"DataLoaders: {PATCH_SIZE}^3 patches with compute_dist=True")
print(f"Starting from Phase 1 best checkpoint.")

In [ ]:
# Phase 2 training: fit_flat_cos (already trained — no warmup to avoid disrupting Phase 1 weights)
PHASE2_LR = LR / 3  # ~3.3e-4
PHASE2_EPOCHS = 30
print(f"Training Phase 2: {PHASE2_EPOCHS} epochs, LR={PHASE2_LR:.1e}, fit_flat_cos")
learn2.fit_flat_cos(PHASE2_EPOCHS, lr=PHASE2_LR)

In [ ]:
# Plot Phase 2 training curves
learn2.recorder.plot_loss()
plt.title("Phase 2: BCE + Dice + clDice + Boundary")
plt.show()

## 9. Head-to-Head: Refinement Model vs Hand-Tuned Post-Processing

The acceptance criterion: refinement model must beat hysteresis+closing+dust on the same probmaps.

In [ ]:
# Load best Phase 2 model (or Phase 1 if Phase 2 didn't improve)
try:
    learn2.load("best_refinement_phase2")
    ref_model = learn2.model.eval()
    phase_used = "Phase 2"
except FileNotFoundError:
    learn.load("best_refinement_phase1")
    ref_model = learn.model.eval()
    phase_used = "Phase 1"

print(f"Loaded best model from {phase_used}")

# Evaluate on all val volumes with probmaps
eval_ids = [vid for vid in val_ids if (PROBMAP_DIR / f"{vid}.npy").exists()]
print(f"Evaluating on {len(eval_ids)} val volumes")

results = []
for i, vid in enumerate(eval_ids):
    prob = np.load(PROBMAP_DIR / f"{vid}.npy").astype(np.float32)
    lbl = tifffile.imread(TRAIN_LBL / f"{vid}.tif")

    # --- Hand-tuned baseline ---
    pred_baseline = postprocess(
        prob, t_low=BASELINE_T_LOW, t_high=BASELINE_T_HIGH,
        z_radius=CLOSING_Z_RADIUS, xy_radius=CLOSING_XY_RADIUS,
        min_size=DUST_MIN_SIZE,
    )

    # --- Refinement model ---
    with torch.no_grad(), torch.amp.autocast("cuda"):
        inp = torch.from_numpy(prob).unsqueeze(0).unsqueeze(0).to(DEVICE)
        logits = ref_model(inp)
        pred_refined = (torch.sigmoid(logits).squeeze().cpu().numpy() > 0.5).astype(np.uint8)

    # --- Score both ---
    ds = METRIC_DOWNSAMPLE
    scale = 1.0 / ds

    pred_b_ds = scipy_zoom(pred_baseline, scale, order=0).astype(np.uint8)
    pred_r_ds = scipy_zoom(pred_refined, scale, order=0).astype(np.uint8)
    lbl_ds = scipy_zoom(lbl, scale, order=0).astype(np.uint8)

    report_b = compute_leaderboard_score(
        pred_b_ds, lbl_ds, ignore_label=2, spacing=(1,1,1),
        surface_tolerance=2.0, voi_alpha=0.3, combine_weights=(0.3, 0.35, 0.35),
    )
    report_r = compute_leaderboard_score(
        pred_r_ds, lbl_ds, ignore_label=2, spacing=(1,1,1),
        surface_tolerance=2.0, voi_alpha=0.3, combine_weights=(0.3, 0.35, 0.35),
    )

    delta = report_r.score - report_b.score
    results.append({
        "vol_id": vid,
        "baseline": report_b.score,
        "refined": report_r.score,
        "delta": delta,
    })
    print(f"  [{i+1}/{len(eval_ids)}] {vid}: baseline={report_b.score:.4f}, "
          f"refined={report_r.score:.4f}, delta={delta:+.4f}")

# Summary
results_df = pd.DataFrame(results)
print(f"\n{'='*60}")
print(f"HEAD-TO-HEAD RESULTS ({len(eval_ids)} val volumes)")
print(f"{'='*60}")
print(f"  Baseline (hand-tuned): {results_df.baseline.mean():.4f} mean comp_score")
print(f"  Refinement model:      {results_df.refined.mean():.4f} mean comp_score")
print(f"  Delta:                 {results_df.delta.mean():+.4f}")
print(f"  Wins: {(results_df.delta > 0).sum()}/{len(results_df)}, "
      f"Losses: {(results_df.delta < 0).sum()}/{len(results_df)}, "
      f"Ties: {(results_df.delta == 0).sum()}/{len(results_df)}")

if results_df.delta.mean() > 0:
    print("\n>>> PASS: Refinement model beats hand-tuned post-processing! <<<")
else:
    print("\n>>> FAIL: Hand-tuned post-processing still wins. <<<")
    print("Consider: more training epochs, different LR, or Phase 2 losses.")

## 10. Kaggle Export

Trace the refinement model with `torch.jit.trace` for Kaggle deployment.
The Kaggle pipeline becomes: main model → Gaussian SWI + logit TTA → probmap → refinement model → threshold → done.

In [ ]:
# Load best model for export
try:
    learn2.load("best_refinement_phase2")
    ref_model = learn2.model.eval().cpu()
    phase_used = "Phase 2"
except FileNotFoundError:
    learn.load("best_refinement_phase1")
    ref_model = learn.model.eval().cpu()
    phase_used = "Phase 1"

print(f"Exporting {phase_used} model...")

# Trace with a dummy 320^3 input
dummy_input = torch.randn(1, 1, VOL_SIZE, VOL_SIZE, VOL_SIZE)
traced = torch.jit.trace(ref_model, dummy_input)

# Verify traced model produces same output
with torch.no_grad():
    out_orig = ref_model(dummy_input)
    out_traced = traced(dummy_input)
    max_diff = (out_orig - out_traced).abs().max().item()
    print(f"Max difference (original vs traced): {max_diff:.2e}")
    assert max_diff < 1e-5, f"Traced model diverges! max_diff={max_diff}"

# Save traced model
traced_path = ROOT / "kaggle" / "kaggle_weights" / "refinement_model_traced.pt"
traced_path.parent.mkdir(parents=True, exist_ok=True)
traced.save(str(traced_path))

size_mb = traced_path.stat().st_size / 1e6
print(f"Saved: {traced_path}")
print(f"Size: {size_mb:.1f} MB")
print(f"\nKaggle usage:")
print(f"  ref_model = torch.jit.load('refinement_model_traced.pt').cuda().eval()")
print(f"  logits = ref_model(probmap.unsqueeze(0).unsqueeze(0).cuda())")
print(f"  pred = (torch.sigmoid(logits).squeeze().cpu().numpy() > 0.5).astype(np.uint8)")

In [ ]:
# Visualize refinement model predictions vs baseline on a val volume
ref_model = ref_model.to(DEVICE).eval()

viz_id = eval_ids[0] if eval_ids else val_ids[0]
prob = np.load(PROBMAP_DIR / f"{viz_id}.npy").astype(np.float32)
lbl = tifffile.imread(TRAIN_LBL / f"{viz_id}.tif")

# Baseline
pred_baseline = postprocess(
    prob, t_low=BASELINE_T_LOW, t_high=BASELINE_T_HIGH,
    z_radius=CLOSING_Z_RADIUS, xy_radius=CLOSING_XY_RADIUS,
    min_size=DUST_MIN_SIZE,
)

# Refinement
with torch.no_grad(), torch.amp.autocast("cuda"):
    inp = torch.from_numpy(prob).unsqueeze(0).unsqueeze(0).to(DEVICE)
    pred_refined = (torch.sigmoid(ref_model(inp)).squeeze().cpu().numpy() > 0.5).astype(np.uint8)

mid = VOL_SIZE // 2
fig, axes = plt.subplots(2, 4, figsize=(20, 10))

for row, (z, title_z) in enumerate([(mid, f"z={mid}"), (mid + 40, f"z={mid+40}")]):
    z = min(z, VOL_SIZE - 1)
    axes[row, 0].imshow(prob[z], cmap="hot", vmin=0, vmax=1)
    axes[row, 0].set_title(f"Probmap {title_z}")

    gt_slice = lbl[z]
    rgb_gt = np.zeros((*gt_slice.shape, 3))
    rgb_gt[gt_slice == 1] = [0, 1, 0]
    axes[row, 1].imshow(rgb_gt)
    axes[row, 1].set_title(f"GT {title_z}")

    axes[row, 2].imshow(pred_baseline[z], cmap="gray")
    axes[row, 2].set_title(f"Baseline {title_z}")

    axes[row, 3].imshow(pred_refined[z], cmap="gray")
    axes[row, 3].set_title(f"Refined {title_z}")

for ax in axes.flat:
    ax.axis("off")
plt.suptitle(f"Volume {viz_id}: Probmap → GT → Baseline → Refined", fontsize=14)
plt.tight_layout()
plt.show()